In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%%html
<style>
.cell-output-ipywidget-background {
    background-color: transparent !important;
}
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}  
</style>

In [ ]:
import asyncio
import json
import random
import re
from typing import TypedDict

from dotenv import load_dotenv

import art
from art.local import LocalBackend

load_dotenv()


class TemporalCluePuzzle(TypedDict):
    num_clues: int
    prompt: str
    solution: dict[str, str]


puzzles_path = "../data/temporal-clue/puzzles.json"
puzzles: list[TemporalCluePuzzle] = json.loads(open(puzzles_path).read())
val_puzzles = puzzles[:64]
test_puzzles = puzzles[64:128]
train_puzzles = puzzles[128:]
random.seed(42)
random.shuffle(train_puzzles)


async def rollout(model: art.Model, puzzle: TemporalCluePuzzle) -> art.Trajectory:
    messages: art.Messages = [
        {"role": "user", "content": puzzle["prompt"] + " /no_think"}
    ]
    client = model.openai_client()
    chat_completion = await client.chat.completions.create(
        messages=messages, model=model.name, max_tokens=4096
    )
    choice = chat_completion.choices[0]
    content = choice.message.content
    assert isinstance(content, str)
    num_correct = 0
    for key, value in puzzle["solution"].items():
        if matches := re.findall(rf"{key}\. ([A-Za-z \.:-]+)", content):
            match = matches[-1]
            if match.strip().lower() == value.lower():
                num_correct += 1
    reward = acc = num_correct / len(puzzle["solution"])
    return art.Trajectory(
        messages_and_choices=[*messages, choice], reward=reward, metrics={"acc": acc}
    )


model = art.TrainableModel(
    name="055", project="temporal-clue", base_model="Qwen/Qwen2.5-7B-Instruct"
)
backend = LocalBackend()
await model.register(backend)

stride = 8

unstarted = list(
    art.gather_trajectory_groups(
        (
            art.TrajectoryGroup(rollout(model, puzzle) for _ in range(16))
            for puzzle in train_puzzles[i * stride : (i + 1) * stride]
        ),
        pbar_desc=f"batch: {i}",
    )
    for i in range(await model.get_step(), len(train_puzzles) // stride)
)
pending: set[asyncio.Task[list[art.TrajectoryGroup]]] = set()
max_pending = 2


def queue_batches() -> None:
    while len(pending) < max_pending and unstarted:
        pending.add(asyncio.create_task(unstarted.pop(0)))


queue_batches()
while pending:
    done, pending = await asyncio.wait(pending, return_when=asyncio.FIRST_COMPLETED)
    queue_batches()
    for task in done:
        if await model.get_step() % 5 == 0:
            val_groups = await art.gather_trajectory_groups(
                (
                    art.TrajectoryGroup(rollout(model, puzzle) for _ in range(1))
                    for puzzle in val_puzzles
                ),
                pbar_desc="val",
                pbar_total_completion_tokens=False,
            )
            await model.log(val_groups)
            queue_batches()
        train_groups = task.result()
        for group in train_groups:
            max_reward = max(trajectory.reward for trajectory in group)
            for trajectory in group:
                trajectory.metrics["max_reward"] = max_reward
        await model.delete_checkpoints()
        await model.train(
            train_groups,
            config=art.TrainConfig(learning_rate=5e-6),
            _config=art.dev.TrainConfig(
                precalculate_logprobs=True, truncated_importance_sampling=10.0
            ),
        )

wandb: Currently logged in as: bradhilton to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


INFO 08-12 20:34:29 [__init__.py:235] Automatically detected platform cuda.


/home/ubuntu/sky_workdir/src/art/__init__.py:10: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth  # type: ignore # noqa: F401


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 08-12 20:34:40 [__init__.py:235] Automatically detected platform cuda.
Unsloth: Patching vLLM v1 graph capture
Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.8.1: Fast Qwen2 patching. Transformers: 4.53.2. vLLM: 0.10.0.
   \\   /|    NVIDIA H100 PCIe. Num GPUs = 1. Max memory: 79.189 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 9.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit with actual GPU utilization = 78.37%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.19 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill toke

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00, 25.69it/s]

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  2.53it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.30it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:01<00:00,  1.40it/s]



...

train:   0%|          | 0/37 [00:00<?, ?it/s]

batch: 33:   0%|          | 0/128 [00:00<?, ?it/s]

Deleted checkpoint /home/ubuntu/sky_workdir/.art/temporal-clue/models/055/checkpoints/0031
Packed 128 trajectories into 34 sequences of length 6144


train:   0%|          | 0/34 [00:00<?, ?it/s]

batch: 34:   0%|          | 0/128 [00:00<?, ?it/s]